[Reference](https://medium.com/@paulhoke/strata-explained-running-125b-parameter-models-on-a-gaming-pc-0a35dec96d8b$0)

```
strata launch --plan --family qwen --model iq2_xs
```

In [1]:
"""
Side-by-side latency check for two OpenAI-compatible local endpoints.

Streams one prompt to each server and reports time to first token and
approximate decode speed, so engines can be compared on the same machine.
"""
import time
import typer

from dataclasses import dataclass
from datetime import datetime
from loguru import logger
from openai import OpenAI

# ============================================================================
# Configuration
# ============================================================================
DEFAULT_PROMPT = "Write a Python function that merges two sorted lists, with tests."
DEFAULT_MAX_TOKENS = 512

app = typer.Typer()

# ============================================================================
# Data Models
# ============================================================================
@dataclass
class StreamResult:
    """Timing results for one streamed completion."""

    label: str
    ttft_seconds: float
    chunks: int
    decode_seconds: float

    @property
    def chunks_per_second(self) -> float:
        """Approximate decode speed in streamed chunks per second."""

        if self.decode_seconds <= 0:
            return 0.0

        return self.chunks / self.decode_seconds

# ============================================================================
# Measurement
# ============================================================================
def measure(label: str, base_url: str, prompt: str, max_tokens: int) -> StreamResult:
    """Stream one completion and time it.
    Args:
        label: Display name for the endpoint.
        base_url: OpenAI-compatible base URL, ending in /v1.
        prompt: User prompt sent to the model.
        max_tokens: Output token cap.
    Returns:
        A StreamResult with first-token latency and decode timing.
    """

    client = OpenAI(base_url=base_url, api_key="local")
    started = time.perf_counter()
    first_token_at: float | None = None
    chunks = 0

    # Thinking is disabled so both engines spend tokens on the answer only
    stream = client.chat.completions.create(
        model="local",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=0,
        stream=True,
        extra_body={"reasoning_effort": "none"},
    )

    for event in stream:
        if not event.choices:
            continue

        delta = event.choices[0].delta

        if not (delta.content or getattr(delta, "reasoning_content", None)):
            continue

        if first_token_at is None:
            first_token_at = time.perf_counter()

        chunks += 1

    finished = time.perf_counter()
    first = first_token_at or finished

    return StreamResult(
        label=label,
        ttft_seconds=round(first - started, 3),
        chunks=chunks,
        decode_seconds=round(finished - first, 3),
    )

@app.command()
def main(
    strata_url: str = typer.Option("http://127.0.0.1:8080/v1", help="Strata base URL"),
    llamacpp_url: str = typer.Option("http://127.0.0.1:8081/v1", help="llama-server base URL"),
    prompt: str = typer.Option(DEFAULT_PROMPT, help="Prompt to send"),
    max_tokens: int = typer.Option(DEFAULT_MAX_TOKENS, help="Output token cap"),
) -> None:
    """Compare first-token latency and decode speed across two engines."""

    run_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    logger.info(f"Run started {run_at}")
    targets = {"strata": strata_url, "llama.cpp": llamacpp_url}

    for label, url in targets.items():
        result = measure(label, url, prompt, max_tokens)

        logger.info(
            f"{result.label}: TTFT {result.ttft_seconds}s, "
            f"{result.chunks} chunks, ~{result.chunks_per_second:.1f} chunks/s"
        )

if __name__ == "__main__":
    app()